# spikes2slope exploratory data analysis

### imports

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from pathlib import Path
import json
from IPython.display import display
from PIL import Image

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
from allensdk.brain_observatory.ecephys.visualization import plot_mean_waveforms, plot_spike_counts, raster_plot
from allensdk.brain_observatory.visualization import plot_running_speed

pd.set_option("display.max_columns", None)

/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### cache setup

In [2]:
data_directory = os.path.expanduser("~/allen_neuropixels_data")

manifest_path = os.path.join(data_directory, "manifest.json")

cache = EcephysProjectCache.from_warehouse(
    manifest=manifest_path
)

In [3]:
sessions = cache.get_session_table()

sessions.head()

,published_at,specimen_id,session_type,age_in_days,sex,full_genotype,unit_count,channel_count,probe_count,ecephys_structure_acronyms
id,,,,,,,,,,
715093703,2019-10-03T00:00:00Z,699733581,brain_observatory_1.1,118.0,M,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,884,2219,6,"[CA1, VISrl, nan, PO, LP, LGd, CA3, DG, VISl, ..."
719161530,2019-10-03T00:00:00Z,703279284,brain_observatory_1.1,122.0,M,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,755,2214,6,"[TH, Eth, APN, POL, LP, DG, CA1, VISpm, nan, N..."
721123822,2019-10-03T00:00:00Z,707296982,brain_observatory_1.1,125.0,M,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,444,2229,6,"[MB, SCig, PPT, NOT, DG, CA1, VISam, nan, LP, ..."
732592105,2019-10-03T00:00:00Z,717038288,brain_observatory_1.1,100.0,M,wt/wt,824,1847,5,"[grey, VISpm, nan, VISp, VISl, VISal, VISrl]"
737581020,2019-10-03T00:00:00Z,718643567,brain_observatory_1.1,108.0,M,wt/wt,568,2218,6,"[grey, VISmma, nan, VISpm, VISp, VISl, VISrl]"


### download and load individual session

In [4]:
session_id = 756029989

session = cache.get_session_data(session_id)

session.probes

/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


,description,location,sampling_rate,lfp_sampling_rate,has_lfp_data
id,,,,,
760640083,probeA,See electrode locations,29999.949611,1249.997900,True
760640087,probeB,See electrode locations,29999.902541,1249.995939,True
760640090,probeC,See electrode locations,29999.905275,1249.996053,True
760640094,probeD,See electrode locations,29999.905275,1249.996053,True
760640097,probeE,See electrode locations,29999.985335,1249.999389,True
760640104,probeF,See electrode locations,30000.023099,1250.000962,True


In [5]:
session.channels.head(3)

/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


,filtering,probe_channel_number,probe_horizontal_position,probe_id,probe_vertical_position,structure_acronym,ecephys_structure_id,ecephys_structure_acronym,anterior_posterior_ccf_coordinate,dorsal_ventral_ccf_coordinate,left_right_ccf_coordinate
id,,,,,,,,,,,
850124396,AP band: 500 Hz high-pass; LFP band: 1000 Hz l...,351,27,760640097,3520,None,NaN,NaN,8069.0,1341.0,9913.0
850124400,AP band: 500 Hz high-pass; LFP band: 1000 Hz l...,353,11,760640097,3540,None,NaN,NaN,8074.0,1327.0,9926.0
850124404,AP band: 500 Hz high-pass; LFP band: 1000 Hz l...,355,27,760640097,3560,None,NaN,NaN,8079.0,1312.0,9939.0


### What brain regions are actually represented by electrode channels in this session, and how many channels do we have in each?

In [6]:
session.channels["ecephys_structure_acronym"].value_counts()

CA1      351
DG       185
APN      125
VISam     93
VISpm     90
SUB       88
VISp      83
VISal     82
CA3       79
LP        74
LGd       74
VISl      73
VISrl     71
SCig      68
TH        45
VPM       24
Eth       22
POL       18
MB        12
POST      10
CA2        8
PO         4
Name: ecephys_structure_acronym, dtype: int64

### all 83 VISp channels are on probe 760640090 (probec)

In [7]:
visp_channels = session.channels[
    session.channels["ecephys_structure_acronym"] == "VISp"
]

visp_channels[[
    "probe_id",
    "probe_channel_number",
    "probe_vertical_position",
    "ecephys_structure_acronym"
]]

,probe_id,probe_channel_number,probe_vertical_position,ecephys_structure_acronym
id,,,,
850125630,760640090,302,3040,VISp
850125636,760640090,305,3060,VISp
850125640,760640090,307,3080,VISp
850125642,760640090,308,3100,VISp
850125646,760640090,310,3120,VISp
...,...,...,...,...
850125612,760640090,293,2940,VISp
850125614,760640090,294,2960,VISp
850125618,760640090,296,2980,VISp


### download and load measured lfp from probec

In [8]:
probe_id = 760640090

lfp = session.get_lfp(probe_id)

lfp

/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


<xarray.DataArray 'LFP' (time: 12453011, channel: 83)>
array([[ 5.2845000e-05,  2.4569999e-05,  3.4320001e-05, ...,
        -6.8250001e-06,  1.7355000e-05, -3.7049999e-06],
       [ 1.5639000e-04,  5.6354998e-05,  1.4449500e-04, ...,
        -1.0335000e-05,  1.2480000e-05,  9.7500003e-07],
       [ 1.2733499e-04,  3.8415001e-05,  8.2095001e-05, ...,
        -1.5600001e-05,  5.0700000e-06,  4.4849999e-06],
       ...,
       [-2.1255000e-05, -1.6067999e-04,  3.0420000e-05, ...,
        -4.5825000e-05, -6.4740001e-05, -9.1649999e-06],
       [ 7.6634999e-05, -7.2540002e-05,  1.2948000e-04, ...,
        -5.2650001e-05, -5.0309998e-05, -6.2399999e-06],
       [ 4.6214998e-05, -5.4600001e-05,  2.6520000e-05, ...,
        -3.1784999e-05, -2.4765000e-05, -7.6050001e-06]], dtype=float32)
Coordinates:
  * time     (time) float64 3.774 3.775 3.776 ... 9.966e+03 9.966e+03 9.966e+03
  * channel  (channel) int64 850125030 850125038 ... 850125678 850125686

LFP is an xarray.DataArray with the following dimensions:

In [9]:
print(lfp.dims)
print(lfp.shape)

('time', 'channel')
(12453011, 83)


In [10]:
opto = session.optogenetic_stimulation_epochs
opto.head()

/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


,start_time,condition,level,stop_time,stimulus_name,duration
id,,,,,,
0,9224.65339,half-period of a cosine wave,1.0,9225.65339,raised_cosine,1.000
1,9226.82361,half-period of a cosine wave,2.5,9227.82361,raised_cosine,1.000
2,9228.64353,a single square pulse,1.0,9228.64853,pulse,0.005
3,9230.65374,a single square pulse,4.0,9230.66374,pulse,0.010
4,9232.42375,2.5 ms pulses at 10 Hz,4.0,9233.42375,fast_pulses,1.000


In [11]:
hard drive download data dandy archive.

start with 1 second epochs, calculate the power section across each epoch, average power spectrum across epochs, fit spec param to the power spectra

fit spec param acloss sliding window. from that generate continous signal, shows aperiodic dynamics

validation important
extract out tagged units. 
later confusion matrix, make sure excitatory are acc excitatory.

SyntaxError: invalid syntax (1568321273.py, line 1)